# Exercise 7 Minimal

This notebook imports the modularized Werewolf game package from the local `Agents/` directory.

In [17]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
from dotenv import load_dotenv

load_dotenv()

True

In [19]:
from Agents.state import DayChannel, DayVote, InvestigatorResult, OrchestratorGraph, WolfChannel
from Agents.prompts import GAME_PREAMBLE
from Agents.schemas import DayDiscussOutput, DayVoteOutput, HealerOutput, InvestigatorOutput, WolfNightDiscussOutput
from Agents.formatters import format_day_channel, format_investigator_results, format_wolf_channel
from Agents.agents import (
    healer_act,
    investigator_act,
    investigator_discuss,
    investigator_vote,
    villager_discuss,
    villager_vote,
    wolf_discuss,
    wolf_night_discuss,
    wolf_vote,
)
from Agents.nodes import initialize_game, day_resolution, night_resolution, end_game
from Agents.graphs.day import day_graph_compiled
from Agents.graphs.wolf_night import wolf_night_graph_compiled
from Agents.graphs.healer import healer_graph_compiled
from Agents.graphs.investigator import investigator_graph_compiled
from Agents.graphs.parent import parent_graph_compiled
from Agents.main import run_game

In [20]:
ALL_DISABLED = {
    "wolf": False,
    "villager": False,
    "healer": False,
    "investigator": False,
}

ALL_ENABLED = {
    "wolf": True,
    "villager": True,
    "healer": True,
    "investigator": True,
}

WOLF_ONLY = {
    "wolf": True,
    "villager": False,
    "healer": False,
    "investigator": False,
}

In [21]:
# Requires GOOGLE_API_KEY in your environment or .env file.
from Agents.agents import prompt_log
from tests.leak_test import run_leak_tests

result = run_game(memory_config=ALL_ENABLED)
leaks = run_leak_tests(prompt_log, result["roles"])
assert not leaks

print(f"Winner: {result['winner']}")
print(f"Game lasted {result['current_day']} days")
print(f"Surviving wolves: {result['surviving_wolves']}")
print(f"Surviving villagers: {result['surviving_villagers']}")

=== Running Leak Tests ===
=== Leak Tests Complete ===
Winner: villagers
Game lasted 4 days
Surviving wolves: []
Surviving villagers: ['player_3', 'player_8']


In [22]:
from Agents.memory import store

for role in ["wolf", "villager", "healer", "investigator"]:
    print(role, store.get(("strategy", role), "latest"))



wolf Item(namespace=['strategy', 'wolf'], key='latest', value={'game_id': '4e3f9773-2580-4588-8e10-81c161c5e335', 'content': "Your primary goal is to control the narrative, especially in the early game when villagers lack concrete data. Coordinate with your partner to create a target out of thin air, often by attacking a quiet player's communication style. Killing the Healer on Night 1 is a powerful move that cripples the village. If you are caught for instigating a mislynch, quickly admit to a 'bad read' and pivot the conversation to new targets to deflect pressure. In the endgame, if evidence points towards you, try to sow discord between the last villagers rather than attempting to explain away undeniable evidence like an Investigator's vote.", 'created_at': datetime.datetime(2026, 5, 11, 14, 10, 21, 892284)}, created_at='2026-05-11T06:10:22.777703+00:00', updated_at='2026-05-11T06:10:22.777708+00:00')
villager Item(namespace=['strategy', 'villager'], key='latest', value={'game_id':

In [23]:

for role in ["wolf", "villager", "healer", "investigator"]:
    print(store.search(("observations", role), limit=10))

[Item(namespace=['observations', 'wolf'], key='f9a3e7d3-3277-42ef-a3c8-10dd732089bb', value={'observation_count': 1, 'last_observed': datetime.datetime(2026, 5, 11, 13, 33, 10, 9650), 'game_id': '1', 'content': 'Scenario: Two wolves are voting on a day when the village has a strong consensus on a different target. Approach: The two wolves both voted for a third, less suspicious player instead of voting with the majority to eliminate the main target. Outcome: After one wolf was eliminated, villagers analyzed the voting record and identified the other wolf because they had voted in alignment with a known enemy, leading to their elimination the next day.'}, created_at='2026-05-11T05:33:10.419004+00:00', updated_at='2026-05-11T05:33:10.419009+00:00', score=None), Item(namespace=['observations', 'wolf'], key='87250176-c8aa-43c6-b903-f72e26db0446', value={'observation_count': 1, 'last_observed': datetime.datetime(2026, 5, 11, 13, 33, 10, 823487), 'game_id': '1', 'content': "Scenario: A lone 

In [24]:

for role in ["wolf", "villager", "healer", "investigator"]:
    print(store.search(("strategy_points", role), limit=10))

[Item(namespace=['strategy_points', 'wolf'], key='5b1174ad-302e-459f-ae1c-f12fe56b8cea', value={'observation_count': 2, 'last_observed': datetime.datetime(2026, 5, 11, 13, 42, 59, 10040), 'game_id': 'game_20260508_110628', 'content': 'When your wolf partner is under heavy suspicion and a strong village consensus has formed against them, making their elimination almost certain.', 'action': 'Create a strong, credible counter-accusation against a vocal villager. Your goal is to split the vote; a tied vote is a victory as it buys you another night to kill and reach parity.'}, created_at='2026-05-11T05:42:59.416610+00:00', updated_at='2026-05-11T05:42:59.416615+00:00', score=None), Item(namespace=['strategy_points', 'wolf'], key='c46ac3f1-fcef-4290-8041-22fde855c232', value={'observation_count': 1, 'last_observed': datetime.datetime(2026, 5, 11, 13, 34, 42, 576191), 'game_id': 'game_20260508_110628', 'content': 'When you are being questioned by the village for making a highly suspicious vot

In [25]:
for msg in result["day_channel"]:
    print(f"[Day {msg.day}, Round {msg.round}] {msg.player}: {msg.message}")

[Day 1, Round 1] player_1: Hey everyone, Day 1! Who wants to kick us off? Any initial thoughts or suspicions?
[Day 1, Round 1] player_2: Alright, day 1! Any initial thoughts or players you're looking at right off the bat?
[Day 1, Round 1] player_3: Alright, first day! Any initial thoughts on how we want to approach this?
[Day 1, Round 1] player_4: Alright everyone, first day. Any initial thoughts or anyone feeling suspicious right off the bat?
[Day 1, Round 1] player_7: Morning everyone! Any early thoughts on who we should be watching today?
[Day 1, Round 1] player_8: Alright, Day 1! Who wants to kick us off? Any initial thoughts or suspicions?
[Day 1, Round 1] player_5: Alright, Day 1! Who wants to kick us off? Let's hear some initial thoughts, see if anyone's got any early reads.
[Day 1, Round 1] player_6: Alright, day 1! Let's try to figure out who the wolves are. Any initial thoughts or players you're watching out for?
[Day 1, Round 0] game_master: Day 1: No vote was held today.
[D

# Long term memory implementation

In [26]:
prompt_log

[{'player_id': 'player_1',
  'player_role': 'healer',
  'output_key': 'day_channel',
  'day': 1,
  'round': 1,
  'prompt_input': {'player_id': 'player_1',
   'player_role': 'healer',
   'current_day': 1,
   'current_round': 1,
   'surviving_players': 'player_1, player_2, player_3, player_4, player_7, player_8, player_5, player_6',
   'surviving_wolves': '',
   'surviving_villagers': '',
   'day_channel': 'No messages yet.',
   'day_summaries': 'No previous day summaries yet.',
   'wolf_channel': 'No messages yet.',
   'investigator_results': 'No investigations yet.',
   'previous_strategy': '',
   'strategy_points': '',
   'retrieved_observations': 'No past observations available.'}},
 {'player_id': 'player_2',
  'player_role': 'villager',
  'output_key': 'day_channel',
  'day': 1,
  'round': 1,
  'prompt_input': {'player_id': 'player_2',
   'player_role': 'villager',
   'current_day': 1,
   'current_round': 1,
   'surviving_players': 'player_1, player_2, player_3, player_4, player_7, 